In [5]:
import pandas as pd

# ============================================================
# Kansas City Chiefs games, 1985–2014
#
# Includes:
#   - Regular-season games
#   - All postseason/playoff games
#
# Excludes:
#   - Preseason games
#
# Output:
#   kansas_city_chiefs_games_1985_2014.csv
# ============================================================

START_YEAR = 1985
END_YEAR = 2014

OUTPUT_FILE = "kansas_city_chiefs_games_1985_2014.csv"

# nflverse games data
URL = "https://github.com/nflverse/nfldata/raw/master/data/games.csv"


# ------------------------------------------------------------
# Download
# ------------------------------------------------------------

print("Downloading NFL schedule data...")

df = pd.read_csv(URL)

print(f"Downloaded {len(df):,} games.")


# ------------------------------------------------------------
# Select the 1985–2014 seasons
# ------------------------------------------------------------

df = df[
    (df["season"] >= START_YEAR) &
    (df["season"] <= END_YEAR)
].copy()


# ------------------------------------------------------------
# Keep regular season and postseason games
#
# game_type values:
#
# REG = regular season
# WC  = Wild Card
# DIV = Divisional playoff
# CON = Conference Championship
# SB  = Super Bowl
#
# PRESEASON games are not included.
# ------------------------------------------------------------

GAME_TYPES = ["REG", "WC", "DIV", "CON", "SB"]

df = df[
    df["game_type"].isin(GAME_TYPES)
].copy()


# ------------------------------------------------------------
# Find games involving Kansas City
# ------------------------------------------------------------

chiefs = df[
    (df["home_team"] == "KC") |
    (df["away_team"] == "KC")
].copy()


# ------------------------------------------------------------
# Determine the opponent
# ------------------------------------------------------------

chiefs["opponent"] = chiefs.apply(
    lambda row:
        row["away_team"]
        if row["home_team"] == "KC"
        else row["home_team"],
    axis=1
)


# ------------------------------------------------------------
# Convert game date
# ------------------------------------------------------------

chiefs["date"] = pd.to_datetime(
    chiefs["gameday"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")


# ------------------------------------------------------------
# Select the requested columns
# ------------------------------------------------------------

result = chiefs[["date", "opponent"]].copy()


# ------------------------------------------------------------
# Sort chronologically
# ------------------------------------------------------------

result["_sort_date"] = pd.to_datetime(
    result["date"],
    errors="coerce"
)

result = result.sort_values(
    "_sort_date"
).drop(
    columns="_sort_date"
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\nChiefs games by season and game type:")

validation = chiefs.groupby(
    ["season", "game_type"]
).size()

print(validation.to_string())


print()
print(f"Total Chiefs games: {len(result)}")


# Check for missing dates
missing_dates = result["date"].isna().sum()

if missing_dates:
    print(
        f"WARNING: {missing_dates} games have missing dates."
    )


# Check for duplicate rows
duplicates = result.duplicated().sum()

if duplicates:
    print(
        f"WARNING: {duplicates} duplicate rows found."
    )


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

result.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8"
)

print()
print(f"Saved to: {OUTPUT_FILE}")


Downloaded 7,548 games.

Chiefs games by season and game type:
season  game_type
1999    REG          16
2000    REG          16
2001    REG          16
2002    REG          16
2003    DIV           1
        REG          16
2004    REG          16
2005    REG          16
2006    REG          16
        WC            1
2007    REG          16
2008    REG          16
2009    REG          16
2010    REG          16
        WC            1
2011    REG          16
2012    REG          16
2013    REG          16
        WC            1
2014    REG          16

Total Chiefs games: 260

Saved to: kansas_city_chiefs_games_1985_2014.csv


In [6]:
import pandas as pd
import requests
from io import StringIO
import time


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

START_YEAR = 1985
END_YEAR = 2014

OUTPUT_FILE = "chiefs_games_1985_2014.csv"

BASE_URL = "https://www.pro-football-reference.com/teams/kan/{year}.htm"

# A normal browser-like user agent helps avoid unnecessary blocking.
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0 Safari/537.36"
    )
}


# ------------------------------------------------------------
# FUNCTION: flatten pandas column names
# ------------------------------------------------------------

def flatten_columns(df):
    """
    PFR tables sometimes have multi-level column headings.
    Convert them into simple strings.
    """
    if isinstance(df.columns, pd.MultiIndex):
        new_columns = []

        for col in df.columns:
            parts = []

            for item in col:
                if pd.notna(item):
                    item = str(item).strip()

                    if item and item not in parts:
                        parts.append(item)

            new_columns.append(" ".join(parts))

        df.columns = new_columns

    else:
        df.columns = [
            str(col).strip()
            for col in df.columns
        ]

    return df


# ------------------------------------------------------------
# FUNCTION: find the schedule table
# ------------------------------------------------------------

def find_schedule_table(tables):
    """
    Find the PFR table containing the Chiefs' schedule.

    We specifically look for Date, Opp, and Tm columns.
    This distinguishes the main schedule table from
    statistical/opponent tables.
    """

    candidates = []

    for table in tables:

        table = flatten_columns(table.copy())

        columns = set(table.columns)

        has_date = any("Date" == c or c.endswith(" Date") for c in columns)
        has_opp = any(c == "Opp" or c.endswith(" Opp") for c in columns)
        has_tm = any(c == "Tm" or c.endswith(" Tm") for c in columns)
        has_week = any(c == "Week" or c.endswith(" Week") for c in columns)

        if has_date and has_opp and has_tm and has_week:
            candidates.append(table)

    if not candidates:
        return None

    # Normally there should be only one appropriate table.
    return candidates[0]


# ------------------------------------------------------------
# FUNCTION: get one season
# ------------------------------------------------------------

def get_season(year):

    url = BASE_URL.format(year=year)

    print(f"Downloading {year}...")

    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    # PFR sometimes puts tables inside HTML comments.
    # pandas.read_html normally handles the visible schedule,
    # but this fallback exposes commented tables if necessary.
    html = response.text

    try:
        tables = pd.read_html(StringIO(html))
    except ValueError:
        tables = []

    schedule = find_schedule_table(tables)

    # --------------------------------------------------------
    # If the normal page did not yield a schedule table,
    # try the PFR game-log page.
    # --------------------------------------------------------

    if schedule is None:

        print(f"  Main schedule table not found; trying game log...")

        gamelog_url = (
            f"https://www.pro-football-reference.com/"
            f"teams/kan/{year}/gamelog"
        )

        response = requests.get(
            gamelog_url,
            headers=HEADERS,
            timeout=30
        )

        response.raise_for_status()

        try:
            tables = pd.read_html(StringIO(response.text))
        except ValueError:
            tables = []

        # Look for tables containing Date and Opp.
        candidates = []

        for table in tables:

            table = flatten_columns(table.copy())
            columns = set(table.columns)

            has_date = any(
                c == "Date" or c.endswith(" Date")
                for c in columns
            )

            has_opp = any(
                c == "Opp" or c.endswith(" Opp")
                for c in columns
            )

            if has_date and has_opp:
                candidates.append(table)

        if not candidates:
            raise RuntimeError(
                f"Could not find a schedule table for {year}"
            )

        # The gamelog page contains both the Chiefs' tables
        # and opponent tables.  We combine them and remove
        # duplicate games below.
        schedule = pd.concat(
            candidates,
            ignore_index=True
        )

    schedule = flatten_columns(schedule)

    # --------------------------------------------------------
    # Locate columns
    # --------------------------------------------------------

    def find_column(possible_names):

        for name in possible_names:
            if name in schedule.columns:
                return name

        for col in schedule.columns:
            for name in possible_names:
                if col.endswith(name):
                    return col

        return None

    date_col = find_column(["Date"])
    opp_col = find_column(["Opp"])

    if date_col is None or opp_col is None:
        raise RuntimeError(
            f"Could not identify Date/Opp columns for {year}. "
            f"Columns were: {list(schedule.columns)}"
        )

    # --------------------------------------------------------
    # Keep only rows representing actual games
    # --------------------------------------------------------

    result = schedule[[date_col, opp_col]].copy()

    result.columns = ["date", "opponent"]

    # Convert dates to datetime.
    result["date"] = pd.to_datetime(
        result["date"],
        errors="coerce"
    )

    # Remove blank rows, totals, etc.
    result = result.dropna(subset=["date"])

    # Make sure these are actually games belonging to the
    # requested season.  This is especially important because
    # postseason games can occur in January of the following
    # calendar year.
    #
    # We don't filter on calendar year here because a 1993
    # playoff game was played in January 1994.
    
    result["season"] = year

    # Clean opponent names.
    result["opponent"] = (
        result["opponent"]
        .astype(str)
        .str.strip()
    )

    # Remove rows where opponent is missing/invalid.
    invalid_opponents = {
        "",
        "nan",
        "NaN",
        "Opp",
        "Opponent"
    }

    result = result[
        ~result["opponent"].isin(invalid_opponents)
    ]

    # --------------------------------------------------------
    # Remove duplicate games
    #
    # The fallback gamelog can contain both the Chiefs' table
    # and the opponent table, which describe the same game.
    # --------------------------------------------------------

    result = result.drop_duplicates(
        subset=["season", "date", "opponent"]
    )

    return result


# ------------------------------------------------------------
# DOWNLOAD ALL SEASONS
# ------------------------------------------------------------

all_seasons = []

for year in range(START_YEAR, END_YEAR + 1):

    try:

        season = get_season(year)

        print(
            f"  Found {len(season)} games for {year}"
        )

        all_seasons.append(season)

    except Exception as e:

        print(
            f"  ERROR downloading {year}: {e}"
        )

    # Be polite to PFR and avoid sending 30 requests
    # simultaneously.
    time.sleep(1)


# ------------------------------------------------------------
# COMBINE EVERYTHING
# ------------------------------------------------------------

if not all_seasons:
    raise RuntimeError(
        "No seasons were successfully downloaded."
    )

games = pd.concat(
    all_seasons,
    ignore_index=True
)


# ------------------------------------------------------------
# SORT
# ------------------------------------------------------------

games = games.sort_values(
    ["season", "date"]
).reset_index(drop=True)


# ------------------------------------------------------------
# KEEP ONLY THE TWO REQUESTED COLUMNS
# ------------------------------------------------------------

games = games[
    ["date", "opponent"]
]


# Format date as YYYY-MM-DD.
games["date"] = games["date"].dt.strftime(
    "%Y-%m-%d"
)


# ------------------------------------------------------------
# SAVE CSV
# ------------------------------------------------------------

games.to_csv(
    OUTPUT_FILE,
    index=False
)


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print()
print("=" * 60)
print("KANSAS CITY CHIEFS 1985–2014")
print("=" * 60)

print()
print(f"Total games: {len(games)}")

print()
print("Games by season:")

# Reconstruct season from the dates for reporting isn't
# reliable because playoff games may be in January of the
# following year, so report using the original season data.
#
# Instead, print counts from each downloaded season.

for season in all_seasons:
    season_year = season["season"].iloc[0]
    print(
        f"{season_year}: {len(season)} games"
    )

print()
print(f"CSV saved as:")
print(OUTPUT_FILE)

print()
print("First 10 rows:")
print(games.head(10).to_string(index=False))

print()
print("Last 10 rows:")
print(games.tail(10).to_string(index=False))


  ERROR downloading 1985: 403 Client Error: Forbidden for url: https://www.pro-football-reference.com/teams/kan/1985.htm
  ERROR downloading 1986: 403 Client Error: Forbidden for url: https://www.pro-football-reference.com/teams/kan/1986.htm
  ERROR downloading 1987: 403 Client Error: Forbidden for url: https://www.pro-football-reference.com/teams/kan/1987.htm
  ERROR downloading 1988: 403 Client Error: Forbidden for url: https://www.pro-football-reference.com/teams/kan/1988.htm
  ERROR downloading 1989: 403 Client Error: Forbidden for url: https://www.pro-football-reference.com/teams/kan/1989.htm
  ERROR downloading 1990: 403 Client Error: Forbidden for url: https://www.pro-football-reference.com/teams/kan/1990.htm
  ERROR downloading 1991: 403 Client Error: Forbidden for url: https://www.pro-football-reference.com/teams/kan/1991.htm


KeyboardInterrupt: 

In [7]:
import pandas as pd
import requests
from io import StringIO
import time


# ============================================================
# SETTINGS
# ============================================================

START_YEAR = 1985
END_YEAR = 2014

# PackerStats team ID for the Kansas City Chiefs
TEAM_ID = 30

OUTPUT_FILE = "chiefs_games_1985_2014.csv"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/140.0 Safari/537.36"
    )
}


# ============================================================
# GET ONE SEASON
# ============================================================

def get_chiefs_season(year):

    url = (
        f"https://www.packerstats.com/"
        f"schedule.php?fid={TEAM_ID}&sid={year}"
    )

    print(f"Downloading {year}...")

    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    # PackerStats uses ordinary HTML tables, so pandas can
    # read them directly.
    tables = pd.read_html(
        StringIO(response.text)
    )

    season_tables = []

    for table in tables:

        # Normalize column names
        table.columns = [
            str(col).strip()
            for col in table.columns
        ]

        # We want tables containing an Opponent column.
        if "Opponent" in table.columns:

            df = table.copy()

            # Add season before combining.
            df["season"] = year

            season_tables.append(df)

    if not season_tables:
        raise RuntimeError(
            f"No schedule tables found for {year}"
        )

    # Combine regular season and postseason tables.
    games = pd.concat(
        season_tables,
        ignore_index=True
    )

    # --------------------------------------------------------
    # Extract the two fields we want
    # --------------------------------------------------------

    games = games[
        ["Date (Day)", "Opponent", "season"]
    ].copy()

    games.columns = [
        "date",
        "opponent",
        "season"
    ]

    # --------------------------------------------------------
    # Clean date
    # --------------------------------------------------------

    games["date"] = pd.to_datetime(
        games["date"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # Clean opponent
    #
    # PackerStats marks away games with "@".
    # We don't need home/away in the final CSV.
    # --------------------------------------------------------

    games["opponent"] = (
        games["opponent"]
        .astype(str)
        .str.strip()
        .str.replace(
            r"^@\s*",
            "",
            regex=True
        )
    )

    # Remove rows that aren't actual games.
    games = games.dropna(
        subset=["date"]
    )

    games = games[
        games["opponent"].notna()
    ]

    games = games[
        games["opponent"].str.len() > 0
    ]

    # --------------------------------------------------------
    # Remove duplicates, just in case
    # --------------------------------------------------------

    games = games.drop_duplicates(
        subset=["season", "date", "opponent"]
    )

    return games


# ============================================================
# DOWNLOAD ALL SEASONS
# ============================================================

all_seasons = []

for year in range(
    START_YEAR,
    END_YEAR + 1
):

    try:

        season = get_chiefs_season(year)

        print(
            f"  Found {len(season)} games"
        )

        all_seasons.append(season)

    except Exception as e:

        print(
            f"  ERROR: {e}"
        )

    # Don't hammer the server.
    time.sleep(1)


# ============================================================
# CHECK THAT WE GOT EVERYTHING
# ============================================================

expected_seasons = END_YEAR - START_YEAR + 1

if len(all_seasons) != expected_seasons:

    successful_years = {
        int(df["season"].iloc[0])
        for df in all_seasons
    }

    missing_years = [
        year
        for year in range(
            START_YEAR,
            END_YEAR + 1
        )
        if year not in successful_years
    ]

    print()
    print(
        "WARNING: Some seasons failed:"
    )
    print(missing_years)


# ============================================================
# COMBINE SEASONS
# ============================================================

games = pd.concat(
    all_seasons,
    ignore_index=True
)


# ============================================================
# SORT
# ============================================================

games = games.sort_values(
    ["season", "date"]
).reset_index(drop=True)


# ============================================================
# REPORT COUNTS BEFORE DROPPING SEASON
# ============================================================

print()
print("=" * 60)
print("KANSAS CITY CHIEFS — 1985 THROUGH 2014")
print("=" * 60)

print()
print("Games by season:")

counts = (
    games
    .groupby("season")
    .size()
)

for year, count in counts.items():

    print(
        f"{int(year)}: {count} games"
    )

print()
print(
    f"TOTAL: {len(games)} games"
)


# ============================================================
# CREATE FINAL TWO-COLUMN DATASET
# ============================================================

final = games[
    ["date", "opponent"]
].copy()

final["date"] = final["date"].dt.strftime(
    "%Y-%m-%d"
)


# ============================================================
# SAVE CSV
# ============================================================

final.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# FINAL REPORT
# ============================================================

print()
print(
    f"Saved to: {OUTPUT_FILE}"
)

print()
print("First 10 rows:")
print(
    final.head(10).to_string(
        index=False
    )
)

print()
print("Last 10 rows:")
print(
    final.tail(10).to_string(
        index=False
    )
)


  ERROR: `Import lxml` failed.  Use pip or conda to install the lxml package.
  ERROR: `Import lxml` failed.  Use pip or conda to install the lxml package.
  ERROR: `Import lxml` failed.  Use pip or conda to install the lxml package.
  ERROR: `Import lxml` failed.  Use pip or conda to install the lxml package.


KeyboardInterrupt: 

In [8]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time


# ============================================================
# SETTINGS
# ============================================================

START_YEAR = 1985
END_YEAR = 2014

TEAM_ID = 30

OUTPUT_FILE = "chiefs_games_1985_2014.csv"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/140.0 Safari/537.36"
    )
}


# ============================================================
# GET ONE SEASON
# ============================================================

def get_chiefs_season(year):

    url = (
        f"https://www.packerstats.com/"
        f"schedule.php?fid={TEAM_ID}&sid={year}"
    )

    print(f"Downloading {year}...")

    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    games = []

    # --------------------------------------------------------
    # Find all HTML tables
    # --------------------------------------------------------

    for table in soup.find_all("table"):

        rows = table.find_all("tr")

        if not rows:
            continue

        # Find the header row.
        header_row = None

        for row in rows:

            headers = [
                cell.get_text(
                    " ",
                    strip=True
                )
                for cell in row.find_all(
                    ["th", "td"]
                )
            ]

            if "Opponent" in headers:
                header_row = row
                break

        if header_row is None:
            continue

        headers = [
            cell.get_text(
                " ",
                strip=True
            )
            for cell in header_row.find_all(
                ["th", "td"]
            )
        ]

        # Find the relevant columns.
        try:
            date_index = headers.index("Date (Day)")
            opponent_index = headers.index("Opponent")
        except ValueError:
            continue

        # ----------------------------------------------------
        # Read game rows
        # ----------------------------------------------------

        for row in rows:

            cells = row.find_all(
                ["td", "th"]
            )

            if not cells:
                continue

            values = [
                cell.get_text(
                    " ",
                    strip=True
                )
                for cell in cells
            ]

            # Need enough cells for both fields.
            if (
                len(values) <= date_index
                or len(values) <= opponent_index
            ):
                continue

            date_text = values[date_index]
            opponent = values[opponent_index].strip()

            # Skip the header itself.
            if date_text == "Date (Day)":
                continue

            # Convert date.
            date = pd.to_datetime(
                date_text,
                errors="coerce"
            )

            if pd.isna(date):
                continue

            # Remove @ used to indicate an away game.
            opponent = opponent.lstrip("@").strip()

            if not opponent:
                continue

            games.append({
                "date": date,
                "opponent": opponent,
                "season": year
            })

    # --------------------------------------------------------
    # Make dataframe
    # --------------------------------------------------------

    if not games:
        raise RuntimeError(
            f"No games found for {year}"
        )

    df = pd.DataFrame(games)

    # Remove accidental duplicates.
    df = df.drop_duplicates(
        subset=[
            "season",
            "date",
            "opponent"
        ]
    )

    return df


# ============================================================
# DOWNLOAD ALL SEASONS
# ============================================================

all_seasons = []

for year in range(
    START_YEAR,
    END_YEAR + 1
):

    try:

        season = get_chiefs_season(year)

        print(
            f"  Found {len(season)} games"
        )

        all_seasons.append(season)

    except Exception as e:

        print(
            f"  ERROR: {e}"
        )

    time.sleep(1)


# ============================================================
# CHECK FOR MISSING SEASONS
# ============================================================

successful_years = {
    int(df["season"].iloc[0])
    for df in all_seasons
}

missing_years = [
    year
    for year in range(
        START_YEAR,
        END_YEAR + 1
    )
    if year not in successful_years
]

if missing_years:

    print()
    print(
        "WARNING — missing seasons:"
    )
    print(missing_years)


# ============================================================
# COMBINE
# ============================================================

if not all_seasons:

    raise RuntimeError(
        "No seasons were successfully downloaded."
    )

games = pd.concat(
    all_seasons,
    ignore_index=True
)


# ============================================================
# SORT
# ============================================================

games = games.sort_values(
    ["season", "date"]
).reset_index(drop=True)


# ============================================================
# REPORT
# ============================================================

print()
print("=" * 60)
print("KANSAS CITY CHIEFS — 1985 THROUGH 2014")
print("=" * 60)

print()
print("Games by season:")

counts = (
    games
    .groupby("season")
    .size()
)

for year, count in counts.items():

    print(
        f"{int(year)}: {count} games"
    )

print()
print(
    f"TOTAL: {len(games)} games"
)


# ============================================================
# CREATE FINAL CSV
# ============================================================

final = games[
    ["date", "opponent"]
].copy()

final["date"] = final["date"].dt.strftime(
    "%Y-%m-%d"
)


# ============================================================
# SAVE
# ============================================================

final.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print()
print(
    f"Saved: {OUTPUT_FILE}"
)

print()
print("First 10 rows:")
print(
    final.head(10).to_string(
        index=False
    )
)

print()
print("Last 10 rows:")
print(
    final.tail(10).to_string(
        index=False
    )
)


  ERROR: No games found for 1985
  ERROR: No games found for 1986
  ERROR: No games found for 1987


KeyboardInterrupt: 